# Hito 2 · Preparación de matrices y plan de comparación

**Aprendizaje Automático Inicial · Reto 01 · INEC GIRS 2024.**  
**Objetivo:** verificar matrices 215/204, justificar características, documentar la comparación K-Means/Ward y PCA. Aquí todavía no se declaran perfiles de municipios ni resultados finales de agrupamiento.

**Importante:** ejecutar desde un clon del repositorio o subir la carpeta `02_hito2` completa. Los códigos de cantón se mantienen fuera de la matriz.


In [ ]:
from pathlib import Path
import csv, math
import numpy as np
import matplotlib.pyplot as plt

# Para Colab: clonar repositorio si aún no se encuentran los archivos.
R = Path.cwd()
if not (R/'02_hito2').exists():
    !git clone https://github.com/2007Mateo/Aprendizaje_Automatico_Reto01.git
    R = Path('Aprendizaje_Automatico_Reto01')
print('Directorio:', R.resolve())


## 1. Verificación de matrículas e incidencias

Cinco indicadores numéricos, denominadores válidos, sin códigos ni nombres en la distancia. Se conserva el registro auditado de 221 y las 17 incidencias.


In [ ]:
def load(rel):
    with (R/rel).open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

base=load('02_hito2/datos/base_preparada_221.csv')
issues=load('02_hito2/datos/incidencias_17.csv')
features=['CSB_cobertura','DIF_proporcion','ORG_actividad','RI_log_p95','BOTADERO_indicador']

def matrix(n):
    rows=load(f'02_hito2/matrices/matriz_{n}.csv')
    keys=load(f'02_hito2/matrices/claves_{n}.csv')
    assert all(list(r)==features for r in rows)
    X=np.array([[float(r[c]) for c in features] for r in rows],dtype=float)
    assert X.shape==(n,5) and np.isfinite(X).all() and len(keys)==n
    return X,keys

X215,k215=matrix(215)
X204,k204=matrix(204)
assert len(base)==221 and len(issues)==17
assert len(set(r['CantonId'] for r in base))==221
assert [x['CantonId'] for x in k215]==[x['CantonId'] for x in base if x['Alternativo_215']=='1']
assert [x['CantonId'] for x in k204]==[x['CantonId'] for x in base if x['Estricto_204']=='1']
print('OK', 'Base auditada:',len(base),'incidencias:',len(issues))
print('Matriz principal:',X215.shape,'Matriz sensibilidad:',X204.shape)
print('Ceros ambiguos se mantienen solo en la principal; CSB>1 NO se recorta.')


## 2. Distribuciones antes de agrupar

La diferencia entre valores cero, no aplicables y ausentes se documentó antes de formar las matrices completas. Los vectores idénticos representan **distintos municipios legítimos**.


In [ ]:
for n,X in [(215,X215),(204,X204)]:
    print(n,'registros; vectores distintos:',np.unique(X,axis=0).shape[0])
    print('Minimos:',dict(zip(features, X.min(axis=0).round(3))))
    print('Maximos:',dict(zip(features, X.max(axis=0).round(3))))
fig,axs=plt.subplots(1,5,figsize=(15,3))
for i,(ax,col) in enumerate(zip(axs,features)):
    ax.hist(X215[:,i],bins=18)
    ax.set_title(col,fontsize=9)
    ax.set_xlabel('Valor preparado')
    ax.set_ylabel('Municipios')
plt.tight_layout();plt.show()


## 3. Matrices y escalado

**Principal:** los cinco indicadores ya preparados, sin estandarización adicional. `RI_log_p95` usa el mismo divisor fijo para los dos escenarios; se conservan atípicos (>1).  
**Sensibilidad de escalado:** un `StandardScaler` ajustado **solo con 215** se aplicaría a 215 y 204 para comparar sobre la misma geometría. Esta sensibilidad no cambia el contenido de la matriz principal.


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler().fit(X215)
X215_z=scaler.transform(X215)
X204_z=scaler.transform(X204)
print('Dimensiones principales:',X215.shape,X204.shape)
print('Dimensiones sensibilidad escala:',X215_z.shape,X204_z.shape)
print('Medias ajuste 215:',scaler.mean_.round(4))
print('Desv. ajuste 215:',scaler.scale_.round(4))
# No exportar X215_z como matriz principal: solo análisis de sensibilidad.


## 4. Diseño exacto de Hito 3 (NO declarar aquí resultados)

- **Métodos:** K-Means y jerárquico aglomerativo Ward (distancia euclidiana) sobre la misma matriz.
- **Valores k:** 2, 3, 4, 5 y 6.
- **K-Means:** semillas 11, 23, 37, 52, 97 y 30 reinicios (`n_init=30`) por semilla.
- **Evaluación:** silueta en cinco dimensiones, tamaños, ARI de estabilidad entre semillas y comparación sobre los 204 códigos compartidos.
- **PCA:** dos componentes sobre la MISMA matriz usada para agrupar; reportar varianza explicada individual y acumulada, componentes y gráfico con etiquetas de cluster.
- **Interpretación:** unidades originales, población como descriptor solamente, sin afirmar una jerarquía oficial de municipios.

El script `03_hito3_preparacion/experimentos_hito3.py` implementa los análisis para el hito siguiente. No elegir k por silueta sola.
